# Installations

In [ ]:
! pip install -U sentence-transformers
! pip install transformers
! pip install sentencepiece
! pip install accelerate -U
! pip install faiss-gpu

# Load libraries

In [ ]:
# Utility and Configuration
import os
import pickle

# Data Science and Computation
import numpy as np
import pandas as pd
import torch

# Machine Learning and NLP
import faiss
from sentence_transformers import SentenceTransformer

# Progress Tracking
from tqdm.auto import tqdm

# Set CUDA visible devices (optional)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Data paths
COMP_LOC = '/kaggle/input/kaggle-llm-science-exam/'

CD60_LOC = '/kaggle/input/60k-data-with-context-v2/'

CD40_LOC = '/kaggle/input/40k-data-with-context-v2/'

# Read data

## Validation

In [ ]:
PATH = CD60_LOC + 'train_with_context2.csv'

valid = pd.read_csv(PATH)

print('Validation data size:', valid.shape )
valid.head()

## 60k dataset

In [ ]:
PATH = CD60_LOC + 'all_12_with_context2.csv'
cd60k = pd.read_csv(PATH)

cd60k = cd60k.drop(columns="source")
cd60k = cd60k.fillna('')

cd60k['source'] = '60k'

print('60k data size:', cd60k.shape )
cd60k.head(2)

## 40k dataset

### MMLU

In [ ]:
# MMLU dataset
PATH = CD40_LOC + 'MMLU_17k_with_context2.csv'
MMLU = pd.read_csv(PATH)


MMLU['E'] = None
keepCols = ['prompt', 'context', 'A', 'B', 'C', 'D', 'E', 'answer']
MMLU = MMLU.loc[MMLU.is_question][keepCols]

MMLU['source'] = 'mmlu'

print('MMLU data size:', MMLU.shape )
MMLU.head(2)

### ScienceQA

In [ ]:
PATH = CD40_LOC + 'ScienceQA_with_context2.csv'
ScienceQA = pd.read_csv(PATH)

keepCols = ['prompt', 'context', 'A', 'B', 'C', 'D', 'E', 'answer']

# ScienceQA_3
ScienceQA_3 = ScienceQA.loc[ScienceQA.image.isna() & (ScienceQA.ct==3)][keepCols]

ScienceQA_3['D'] = None
ScienceQA_3['E'] = None

ScienceQA_3['source'] = 'scienceQA'

# ScienceQA_4
ScienceQA_4 = ScienceQA.loc[ScienceQA.image.isna() & (ScienceQA.ct==4)][keepCols]

ScienceQA_4['E'] = None

ScienceQA_4['source'] = 'scienceQA'

print('ScienceQA_3 data size:', ScienceQA_3.shape )
ScienceQA_3.head(2)

In [ ]:
print('ScienceQA_4 data size:', ScienceQA_4.shape )
ScienceQA_4.head(2)

### OpenBook

In [ ]:
PATH = CD40_LOC + 'OpenBook_with_context2.csv'
OpenBook = pd.read_csv(PATH)
OpenBook = OpenBook.loc[OpenBook.is_question]

# Process OpenBook data
OpenBook.drop(columns = ['id', 'is_question'], inplace = True)
OpenBook['E'] = None
OpenBook = OpenBook[[col for col in OpenBook.columns if col not in ['answer']] + ['answer']]

OpenBook['source'] = 'OpenBook'

print('OpenBook data size:', OpenBook.shape )
OpenBook.head(2)

# 100k dataset (60k + 40k)

In [ ]:
cd100k_df = pd.concat([cd60k, MMLU, ScienceQA_3, ScienceQA_4, OpenBook], ignore_index=True)

cd100k_df.drop_duplicates(inplace=True)
cd100k_df = cd100k_df.reset_index(drop=True)

print('100k data size:', cd100k_df.shape)
cd100k_df.head(2)

# Build FAISS index

In [ ]:
# Sentence Embedding

model = SentenceTransformer('sentence-transformers/paraphrase-albert-small-v2')

sentence_embeddings = model.encode(cd100k_df['prompt'])
dim = sentence_embeddings.shape[1]

sentence_embeddings.shape

In [ ]:
# Build FAISS index
index = faiss.IndexFlatL2(dim)
index.add(sentence_embeddings)

# Remove rows similar to Competition data

## Example

In [ ]:
id = 0

print("-"*25)
print("Question from validation set")
print(valid['prompt'][id])
print("-"*25)

n = 5
query = model.encode([valid['prompt'][id]])

scores, inds = index.search(query, n)

for i, ind in enumerate(inds[0]):
    print(f"Score {scores[0][i]} -  {cd100k_df.iloc[ind]['prompt']}")
    print()

That first one definitely seems like it should be removed from our training set. I will use a threshold score of 100 

## Remove samples from train set

In [ ]:
%%capture
remLst = []

# Score threshold
threshold = 100

# Number of similar questions
n = 20

for prompt in tqdm(valid['prompt'].tolist()):

    query = model.encode([prompt])

    scores, inds = index.search(query, n)

    for i in range(0,len(scores[0])):
        if scores[0][i] < threshold:
            remLst.append(inds[0][i])

In [ ]:
remLst = list(set(remLst))
print(f"Samples to remove: {len(remLst)}")

In [ ]:
cd100k_df = cd100k_df.loc[[i for i in range(0,len(cd100k_df)) if i not in remLst]]
cd100k_df = cd100k_df.reset_index(drop=True)
cd100k_df.drop_duplicates(inplace=True)
cd100k_df.shape

# Manage empty options

In [ ]:
cd100k_df.isnull().sum()

In [ ]:
print("Number of A options empty: ", len(cd100k_df[cd100k_df['A']=='']))
print("Number of B options empty: ", len(cd100k_df[cd100k_df['B']=='']))
print("Number of C options empty: ", len(cd100k_df[cd100k_df['C']=='']))
print("Number of D options empty: ", len(cd100k_df[cd100k_df['D']=='']))
print("Number of E options empty: ", len(cd100k_df[cd100k_df['E']=='']))

In [ ]:
cd100k_df['source'].value_counts()

In [ ]:
cd100k_df = cd100k_df[(cd100k_df['A']!='') &
                      (cd100k_df['B']!='') &
                      (cd100k_df['C']!='') &
                      (cd100k_df['D']!='') &
                      (cd100k_df['E']!='') 
                      ]

cd100k_df = cd100k_df.reset_index(drop=True)
cd100k_df.drop_duplicates(inplace=True)

In [ ]:
print("Number of A options empty: ", len(cd100k_df[cd100k_df['A']=='']))
print("Number of B options empty: ", len(cd100k_df[cd100k_df['B']=='']))
print("Number of C options empty: ", len(cd100k_df[cd100k_df['C']=='']))
print("Number of D options empty: ", len(cd100k_df[cd100k_df['D']=='']))
print("Number of E options empty: ", len(cd100k_df[cd100k_df['E']=='']))

In [ ]:
cd100k_df['source'].value_counts()

# Rebuild FAISS index

In [ ]:
sentence_embeddings = model.encode(cd100k_df['prompt'])
dim = sentence_embeddings.shape[1]

print(sentence_embeddings.shape)

# Build FAISS index
index = faiss.IndexFlatL2(dim)
index.add(sentence_embeddings)

# Non-Science questions

In [ ]:
id = 20

print("-"*25)
print(valid['prompt'][id])
print("-"*25)

n = 10000
query = model.encode([valid['prompt'][id]])

scores, inds = index.search(query, n)

ind = inds[0][-3:]
print(cd100k_df.iloc[ind]['prompt'].values)
print()

These definitely don't seem relevant to our Science Exam!

# Find Science questions

## Example

In [ ]:
id = 0

print("-"*25)
print(valid['prompt'][id])
print("-"*25)

n = 10
query = model.encode([valid['prompt'][id]])

scores, inds = index.search(query, n)

for ind in inds[0]:
    print(cd100k_df.iloc[ind]['prompt'])
    print()

I'm no scientist but those look alright to me

## Find questions

In [ ]:
%%capture

indLst = []

# Number of similar questions
# Tweak this number to build larger science dataset
NUM_Qs = 7

for prompt in tqdm(valid['prompt'].tolist()):

    query = model.encode([prompt])

    scores, inds = index.search(query, NUM_Qs)

    for i in range(0,len(scores[0])):
        indLst.append(inds[0][i])



In [ ]:
indLst = list(set(indLst))
print(f"Samples: {len(indLst)}")

# Science dataset

In [ ]:
final_df = cd100k_df.loc[indLst]
final_df = final_df.reset_index(drop=True)
final_df.drop_duplicates(inplace=True)

print(final_df.shape)
final_df.head(2)

In [ ]:
final_df['source'].value_counts()

In [ ]:
final_df.head(2)

In [ ]:
final_df.isnull().sum()

In [ ]:
print("Number of A options empty: ", len(final_df[final_df['A']=='']))
print("Number of B options empty: ", len(final_df[final_df['B']=='']))
print("Number of C options empty: ", len(final_df[final_df['C']=='']))
print("Number of D options empty: ", len(final_df[final_df['D']=='']))
print("Number of E options empty: ", len(final_df[final_df['E']=='']))

# OpenAI API setup

In [ ]:
! pip install llm

In [ ]:
import llm

model = llm.get_model("gpt-3.5-turbo")

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['OPENAI_API_KEY'] = secrets.get_secret("openai")

model.key = os.environ['OPENAI_API_KEY']

# Generate missing options

In [ ]:
final_df.isnull().sum()

In [ ]:
print("Number of A options empty: ", len(final_df[final_df['A']=='']))
print("Number of B options empty: ", len(final_df[final_df['B']=='']))
print("Number of C options empty: ", len(final_df[final_df['C']=='']))
print("Number of D options empty: ", len(final_df[final_df['D']=='']))
print("Number of E options empty: ", len(final_df[final_df['E']=='']))

In [ ]:
def generate_prompt_optE(row):
    return f'''

            Given the following multiple choice question, please generate an option E in the same format as the other options, which would be plausible but ultimately the wrong answer.
            Return only the option E

            Question: {row.prompt}

            A: {row.A}
            B: {row.B}
            C: {row.C}
            D: {row.D}

            Answer: {row.answer}

            '''

In [ ]:
# Example prompt
example_prompt = generate_prompt_optE(final_df[final_df['E'].isnull()].iloc[0])
print(example_prompt)

In [ ]:
response = model.prompt(example_prompt)
print(response)

In [ ]:
# Option E is empty
E = []

for row in tqdm(final_df.itertuples(), total = len(final_df)):

    # if row.Index < 5466:
    #     continue

    if row.E is None or row.E == '':

        response = model.prompt(generate_prompt_optE(row))
        optE = response.text()
        optE = optE.split(":")[1].strip()

        E.append(optE)

    else:
        E.append(None)



In [ ]:
# Option E
final_df['gen_E'] = E
print("Number of E options generated: ", len(final_df[final_df['gen_E'].notnull()]))

final_df['E'] = np.where((final_df['E'].isnull()) | (final_df['E'] == ''), final_df['gen_E'], final_df['E'])
final_df.drop(columns = ['gen_E'], inplace = True)

# Reorder columns
final_df = final_df[[col for col in final_df.columns if 'answer' not in col] + ['answer']]

final_df.head(2)

In [ ]:
final_df['source'].value_counts()

In [ ]:
final_df = final_df[final_df['D'].notnull()]

final_df.drop_duplicates(inplace = True)
final_df.dropna(inplace=True)
final_df.drop(columns=['source'], inplace=True)
final_df = final_df.reset_index(drop = True)

final_df.shape

In [ ]:
print("Number of A options empty: ", len(final_df[final_df['A']=='']))
print("Number of B options empty: ", len(final_df[final_df['B']=='']))
print("Number of C options empty: ", len(final_df[final_df['C']=='']))
print("Number of D options empty: ", len(final_df[final_df['D']=='']))
print("Number of E options empty: ", len(final_df[final_df['E']=='']))

In [ ]:
final_df.isnull().sum()

# Shuffle options

In [ ]:
import random

In [ ]:
# Function to shuffle columns A, B, C, D, E and update answer accordingly
def shuffle_row(row):
    options = ['A', 'B', 'C', 'D', 'E']
    shuffle_options = ['A', 'B', 'C', 'D', 'E']
    old_answer = row['answer']
    old_answer_value = row[old_answer]

    random.shuffle(shuffle_options)

    new_row = {
                'prompt': row['prompt'],
                'context': row['context'],
                'answer': options[shuffle_options.index(old_answer)],  # Update the answer key
                }

    for opt1, opt2 in zip(['A', 'B', 'C', 'D', 'E'], shuffle_options):
        new_row[opt1] = row[opt2]

    return pd.Series(new_row)

In [ ]:
# Shuffle each row
final_df_shuffled = final_df.apply(shuffle_row, axis=1)

# Reorder columns
final_df_shuffled = final_df_shuffled[final_df.columns]

final_df_shuffled.shape

In [ ]:
final_df_shuffled.drop_duplicates(inplace = True)
final_df_shuffled = final_df_shuffled.reset_index(drop = True)
final_df_shuffled.shape

In [ ]:
final_df_shuffled.isnull().sum()

In [ ]:
print("Number of A options empty: ", len(final_df_shuffled[final_df_shuffled['A']=='']))
print("Number of B options empty: ", len(final_df_shuffled[final_df_shuffled['B']=='']))
print("Number of C options empty: ", len(final_df_shuffled[final_df_shuffled['C']=='']))
print("Number of D options empty: ", len(final_df_shuffled[final_df_shuffled['D']=='']))
print("Number of E options empty: ", len(final_df_shuffled[final_df_shuffled['E']=='']))

In [ ]:
final_df.loc[2]

In [ ]:
final_df_shuffled.loc[2]

# Save dataset

In [ ]:
final_df_shuffled = final_df_shuffled.replace(np.NaN, '')

final_df_shuffled['A'] = final_df_shuffled['A'].map(str)
final_df_shuffled['B'] = final_df_shuffled['B'].map(str)
final_df_shuffled['C'] = final_df_shuffled['C'].map(str)
final_df_shuffled['D'] = final_df_shuffled['D'].map(str)
final_df_shuffled['E'] = final_df_shuffled['E'].map(str)

final_df_shuffled.shape

In [ ]:
final_df_shuffled.to_csv("LLMSE_1k.csv", index=False)